[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/35_reinforcement_learning/01_mdp_and_bellman.ipynb)

# Bài 01 — MDP, phương trình Bellman và quy hoạch động

**🎯 Mục tiêu**
- Khung học tăng cường: tác tử, môi trường, trạng thái, hành động, phần thưởng, **return**, chính sách — khác học có giám sát ở đâu.
- **Khám phá vs khai thác** trên bài toán k cần gạt: tham lam, ε-greedy, UCB.
- **MDP** và **phương trình Bellman**: đánh giá chính sách (lặp và giải hệ tuyến tính — hai cách phải khớp).
- **Value iteration** và **policy iteration** trên FrozenLake (có trượt); ảnh hưởng của độ trượt và hệ số chiết khấu $\gamma$.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/35_reinforcement_learning

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.envs import ACTION_NAMES, GridWorld, KArmedBandit
from ml_course.visualization import set_style

set_style()

## 1. Học tăng cường là gì?

Tác tử (agent) quan sát **trạng thái** $s_t$, chọn **hành động** $a_t$, môi trường trả về **phần thưởng** $r_{t+1}$ và trạng thái mới $s_{t+1}$. Mục tiêu: chọn hành động để cực đại **return** — tổng phần thưởng (chiết khấu) về sau:

$$G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \dots, \qquad 0 \le \gamma \le 1$$

Khác học có giám sát:
- **Không có đáp án đúng** cho từng bước — chỉ có phần thưởng, thường **trễ** (nước cờ hôm nay quyết định thắng thua nhiều nước sau: bài toán *credit assignment*).
- Dữ liệu **phụ thuộc vào chính hành động** của tác tử — phải **khám phá** để biết hành động nào tốt.
- Ứng dụng: game (AlphaGo), robot, điều khiển, hệ gợi ý tối ưu dài hạn, quản lý tài nguyên, và tinh chỉnh LLM theo phản hồi người (RLHF).

## 2. Khám phá vs khai thác: bài toán k cần gạt

10 máy đánh bạc, máy $a$ trả thưởng $\sim\mathcal N(q_*(a), 1)$ với $q_*(a)$ chưa biết. Mỗi lượt chọn một máy. Ước lượng giá trị bằng trung bình mẫu $Q(a)$:
- **Tham lam**: luôn chọn $\arg\max Q(a)$ — có thể kẹt ở máy kém vì không thử máy khác.
- **ε-greedy**: với xác suất $\varepsilon$ chọn ngẫu nhiên.
- **UCB**: chọn $\arg\max\left[Q(a) + c\sqrt{\ln t / N(a)}\right]$ — cộng "phần thưởng cho sự không chắc chắn" vào máy ít được thử.

300 lần chạy, mỗi lần một bộ máy khác nhau, 1 000 lượt:

In [ ]:
def run_bandit(method, steps=1000, runs=300, eps=0.1, c=2.0):
    avg_reward, pct_optimal = np.zeros(steps), np.zeros(steps)
    for run in range(runs):
        bandit = KArmedBandit(10, seed=run)
        rng = np.random.default_rng(1000 + run)
        Q, N = np.zeros(10), np.zeros(10)
        best = bandit.q_star.argmax()
        for t in range(steps):
            if method == "tham lam":
                a = int(Q.argmax())
            elif method == "ε-greedy (ε = 0.1)":
                a = int(rng.integers(10)) if rng.random() < eps else int(Q.argmax())
            else:                                                       # UCB: thử mỗi máy một lần trước
                a = int(np.argmin(N)) if N.min() == 0 else int(np.argmax(Q + c * np.sqrt(np.log(t + 1) / N)))
            r = bandit.pull(a)
            N[a] += 1
            Q[a] += (r - Q[a]) / N[a]                                   # trung bình tăng dần
            avg_reward[t] += r
            pct_optimal[t] += a == best
    return avg_reward / runs, pct_optimal / runs


t0 = time.perf_counter()
bandit_results = {m: run_bandit(m) for m in ["tham lam", "ε-greedy (ε = 0.1)", "UCB (c = 2)"]}
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for name, (rew, opt) in bandit_results.items():
    axes[0].plot(np.convolve(rew, np.ones(20) / 20, mode="valid"), label=name)
    axes[1].plot(np.convolve(opt, np.ones(20) / 20, mode="valid"), label=name)
    print(f"{name:20s}: thưởng trung bình (500 lượt cuối) = {rew[500:].mean():.3f} | chọn đúng máy tốt nhất (100 lượt cuối) = {opt[-100:].mean():.0%}")
axes[0].set_ylabel("thưởng trung bình")
axes[1].set_ylabel("% chọn máy tốt nhất")
for ax in axes:
    ax.set_xlabel("lượt")
    ax.legend()
plt.tight_layout()
plt.show()
print(f"{time.perf_counter() - t0:.0f}s")

Tham lam thường "chốt" sớm vào một máy khá-tốt và không bao giờ phát hiện máy tốt nhất. ε-greedy khám phá ngẫu nhiên — tốt hơn nhiều nhưng vẫn phí 10% lượt vào máy đã biết là kém. UCB khám phá **có định hướng**: thử những máy còn không chắc chắn, nên vừa tìm đúng máy tốt nhất thường xuyên hơn vừa ít lãng phí. Bài toán bandit là nền của A/B testing thích ứng và hệ gợi ý (chọn nội dung để thử).

## 3. MDP: Markov Decision Process

Bandit không có **trạng thái** — hành động không ảnh hưởng tương lai. MDP $(\mathcal S, \mathcal A, P, R, \gamma)$ thêm trạng thái: $P(s', r\mid s, a)$ là xác suất sang $s'$ và nhận $r$ khi làm $a$ ở $s$. **Tính Markov**: tương lai chỉ phụ thuộc trạng thái hiện tại, không phụ thuộc lịch sử.

Môi trường: **FrozenLake** 4×4 — đi từ S tới G (thưởng 1) trên mặt băng, tránh hố H (kết thúc, thưởng 0). Mặt băng **trơn**: với xác suất `slip`, tác tử trượt sang một trong hai hướng vuông góc. Mô hình $P$ cho sẵn đầy đủ: `env.P[s][a]` = danh sách (xác suất, $s'$, $r$, kết thúc).

In [ ]:
env = GridWorld("4x4", slip=2 / 3)                                     # như FrozenLake "is_slippery": 1/3 đúng hướng
print("\n".join(" ".join(row) for row in env.grid))
print(f"\n{env.n_states} trạng thái, {env.n_actions} hành động {ACTION_NAMES}")
print("P[14][→] (ô cạnh G, đi phải):", [(round(p, 3), s2, r, d) for p, s2, r, d in env.P[14][2]])

## 4. Phương trình Bellman và đánh giá chính sách

**Hàm giá trị** của chính sách $\pi$: $V^\pi(s) = \mathbb E_\pi[G_t\mid s_t = s]$. Tách bước đầu tiên ra cho **phương trình Bellman kỳ vọng**:

$$V^\pi(s) = \sum_a\pi(a\mid s)\sum_{s', r}P(s', r\mid s, a)\left[r + \gamma V^\pi(s')\right]$$

Đây là hệ **tuyến tính** $|\mathcal S|$ ẩn: $V = R^\pi + \gamma P^\pi V \Rightarrow V = (I - \gamma P^\pi)^{-1}R^\pi$. Hoặc giải lặp: áp dụng vế phải liên tục (**iterative policy evaluation**) — hội tụ vì đó là ánh xạ co với $\gamma < 1$. Thử với chính sách **ngẫu nhiên đều**:

In [ ]:
GAMMA = 0.99


def expected_reward_and_transition(env, policy):
    """policy: mảng (S, A) xác suất. Trả về R^π (S,) và P^π (S, S) (bỏ chuyển sang trạng thái kết thúc)."""
    R, P = np.zeros(env.n_states), np.zeros((env.n_states, env.n_states))
    for s in range(env.n_states):
        for a in range(env.n_actions):
            for p, s2, r, done in env.P[s][a]:
                R[s] += policy[s, a] * p * r
                if not done:
                    P[s, s2] += policy[s, a] * p
    return R, P


def evaluate_policy_iterative(env, policy, gamma=GAMMA, tol=1e-10):
    V, sweeps = np.zeros(env.n_states), 0
    while True:
        V_new = np.array([sum(policy[s, a] * p * (r + gamma * V[s2] * (not done))
                              for a in range(env.n_actions) for p, s2, r, done in env.P[s][a]) for s in range(env.n_states)])
        sweeps += 1
        if np.abs(V_new - V).max() < tol:
            return V_new, sweeps
        V = V_new


random_policy = np.full((env.n_states, env.n_actions), 0.25)
V_iter, sweeps = evaluate_policy_iterative(env, random_policy)
R_pi, P_pi = expected_reward_and_transition(env, random_policy)
V_exact = np.linalg.solve(np.eye(env.n_states) - GAMMA * P_pi, R_pi)
print(f"lặp: {sweeps} vòng | sai lệch so với nghiệm hệ tuyến tính: {np.abs(V_iter - V_exact).max():.1e}")
print("V^π (chính sách ngẫu nhiên):\n", V_exact.reshape(4, 4).round(3))

Đi ngẫu nhiên, xác suất tới đích từ S rất nhỏ. Câu hỏi tiếp theo: chính sách **tốt nhất** là gì?

## 5. Bellman tối ưu: value iteration và policy iteration

Giá trị tối ưu thoả **phương trình Bellman tối ưu** — thay trung bình theo chính sách bằng **max**:

$$V^*(s) = \max_a\sum_{s', r}P(s', r\mid s, a)\left[r + \gamma V^*(s')\right], \qquad \pi^*(s) = \arg\max_a Q^*(s, a)$$

- **Value iteration**: lặp vế phải tới hội tụ, rồi lấy chính sách tham lam theo $V^*$.
- **Policy iteration**: luân phiên *đánh giá* chính sách hiện tại (giải chính xác) và *cải thiện* (tham lam theo $V^\pi$) — thường hội tụ sau rất ít vòng.

In [ ]:
def q_from_v(env, V, gamma=GAMMA):
    return np.array([[sum(p * (r + gamma * V[s2] * (not done)) for p, s2, r, done in env.P[s][a]) for a in range(env.n_actions)]
                     for s in range(env.n_states)])


def value_iteration(env, gamma=GAMMA, tol=1e-10):
    V, sweeps = np.zeros(env.n_states), 0
    while True:
        V_new = q_from_v(env, V, gamma).max(1)
        sweeps += 1
        if np.abs(V_new - V).max() < tol:
            return V_new, q_from_v(env, V_new, gamma).argmax(1), sweeps
        V = V_new


def policy_iteration(env, gamma=GAMMA):
    pi, rounds = np.zeros(env.n_states, dtype=int), 0
    while True:
        rounds += 1
        R, P = expected_reward_and_transition(env, np.eye(env.n_actions)[pi])
        V = np.linalg.solve(np.eye(env.n_states) - gamma * P, R)                  # đánh giá chính xác
        new_pi = q_from_v(env, V, gamma).argmax(1)                                # cải thiện
        if np.array_equal(new_pi, pi):
            return V, pi, rounds
        pi = new_pi


def success_rate(env, pi, episodes=2000):
    wins, lengths = 0, []
    for ep in range(episodes):
        s, done, t = env.reset(seed=ep), False, 0
        while not done:
            s, r, done = env.step(pi[s])
            t += 1
        wins += r
        lengths.append(t)
    return wins / episodes, np.mean(lengths)


def show_policy(env, pi):
    return "\n".join(" ".join(ACTION_NAMES[pi[r * env.n_cols + c]] if env.grid[r, c] in "SF" else env.grid[r, c]
                              for c in range(env.n_cols)) for r in range(env.n_rows))


for slip in [0.0, 2 / 3]:
    e = GridWorld("4x4", slip=slip)
    V_vi, pi_vi, sweeps = value_iteration(e)
    V_pi, pi_pi, rounds = policy_iteration(e)
    win, length = success_rate(e, pi_vi)
    print(f"── slip = {slip:.2f}: value iteration {sweeps} vòng, policy iteration {rounds} vòng | "
          f"|V_VI − V_PI| = {np.abs(V_vi - V_pi).max():.1e} | V*(S) = {V_vi[0]:.3f}")
    print(f"   mô phỏng 2 000 lượt: thắng {win:.0%}, trung bình {length:.1f} bước")
    print(show_policy(e, pi_vi))

- **Không trơn**: chính sách tối ưu là đường ngắn nhất (6 bước), thắng 100%; $V^*(S) = 0.99^5 \approx 0.95$ — chiết khấu làm phần thưởng ở xa có giá trị nhỏ hơn.
- **Trơn** (chỉ 1/3 đi đúng hướng): chính sách tối ưu trông "kỳ lạ" — ở ô bắt đầu lại hướng **sang trái (vào tường)**: đi vào tường thì trượt chỉ có thể xuống dưới hoặc đứng yên, **không bao giờ** rơi vào hố. Tác tử chấp nhận đi lâu hơn nhiều để an toàn. Không ai lập trình quy tắc "tránh mép hố" — nó tự nảy sinh từ phương trình Bellman.
- Hai thuật toán cho cùng $V^*$; policy iteration chỉ cần vài vòng (mỗi vòng giải một hệ tuyến tính), value iteration cần hàng trăm vòng rẻ.

### Ảnh hưởng của γ

In [ ]:
env8 = GridWorld("8x8", slip=2 / 3)
for gamma in [0.9, 0.99, 0.999]:
    V8, pi8, _ = value_iteration(env8, gamma=gamma)
    win, length = success_rate(env8, pi8, episodes=1000)
    print(f"FrozenLake 8×8 trơn, γ = {gamma}: V*(S) = {V8[0]:.3f} | thắng {win:.0%} | trung bình {length:.0f} bước")

$\gamma$ nhỏ: phần thưởng xa bị chiết khấu mạnh → tác tử "thiếu kiên nhẫn", chọn đường nhanh nhưng rủi ro hơn → thắng ít hơn. $\gamma$ gần 1: sẵn sàng đi vòng rất lâu để an toàn. $\gamma$ vì thế vừa là đặc điểm bài toán (lãi suất, xác suất "trò chơi kết thúc") vừa là siêu tham số định hình hành vi.

## 6. Giới hạn của quy hoạch động

Value/policy iteration cần **biết mô hình** $P$ và duyệt **mọi trạng thái** mỗi vòng. Thực tế: mô hình thường chưa biết (robot, thị trường, người dùng) và không gian trạng thái khổng lồ (cờ vây ~$10^{170}$ trạng thái, ảnh camera là liên tục). Các bài sau: học từ **trải nghiệm** (bài 02–03: Monte Carlo, TD, Q-learning) và **xấp xỉ hàm** bằng mạng nơ-ron (bài 04: DQN, policy gradient).

## 🧠 Kiểm tra nhanh

1. Return là gì? Vai trò của γ?
2. Vì sao chính sách tham lam thất bại trong bài toán bandit? UCB khắc phục thế nào?
3. Viết phương trình Bellman kỳ vọng. Vì sao có thể giải bằng hệ tuyến tính?
4. Value iteration và policy iteration khác nhau thế nào?
5. Vì sao chính sách tối ưu trên băng trơn lại đi vào tường?
6. Quy hoạch động cần những giả định nào?

## 📌 Tóm tắt

- RL: học hành vi từ phần thưởng (trễ), phải cân bằng khám phá–khai thác; bandit là trường hợp không trạng thái (ε-greedy, UCB).
- MDP + tính Markov; $V^\pi$ thoả phương trình Bellman tuyến tính; $V^*$ thoả Bellman tối ưu (max).
- Value iteration / policy iteration tìm chính sách tối ưu khi biết mô hình; hành vi "khôn ngoan" (tránh rủi ro) tự nảy sinh.
- γ định hình mức "kiên nhẫn"; quy hoạch động không mở rộng được khi không biết mô hình hoặc trạng thái quá nhiều.

## 📚 Tài liệu tham khảo

- Sutton & Barto (2018) — *Reinforcement Learning: An Introduction* (2nd ed.), chương 2–4 ([miễn phí](http://incompleteideas.net/book/the-book-2nd.html))
- Bellman (1957) — *Dynamic Programming*
- Auer, Cesa-Bianchi & Fischer (2002) — *Finite-time Analysis of the Multiarmed Bandit Problem* (UCB)
- David Silver — *RL Course* (UCL), bài 1–3
- [Gymnasium — FrozenLake](https://gymnasium.farama.org/environments/toy_text/frozen_lake/)